# 🧠 Meningioma Modelling Notebook

Run from the **repo root** (`meningioma-atypier/`). Consumes `output/datasets/` from the cleaning notebook.

EDA on **unimputed** data. Multivariable modelling on **imputed** data. DDA lives in the cleaning notebook.

<details>
<summary><b>Pipeline map</b> — notebook step → module</summary>

| Step | What it does | Driven by |
|------|--------------|-----------|
| 00 | Setup — imports, keep `output/` | `config/` loader + phase modules |
| 01 | Load handoff (data + schema) | `dataset_handoff.load_modelling_handoff` |
| 02 | EDA / model variant lists | `config/analysis.py` |
| 03 | EDA + diagnostic accuracy | `eda` · `diagnostic_accuracy` |
| 04 | Multivariable logistic (Rubin pool) | `inferential` → `output/inferential/` (incl. `model_artifacts/`) |
| 05 | HTML report | `config/report_settings.py` · `report` |

Config modules live in `config/` (loaded via `load("name")`).

</details>


## 00. Setup

⚙️ Loads modelling modules. Does **not** wipe `output/` — reads cleaning handoff artifacts.

<details>
<summary>🔧 How it works</summary>

- 📦 `from heavy_machinery.config import load` plus `heavy_machinery.cleaning_phase.*` and `heavy_machinery.modelling_phase.*` imports.
- 📁 `OUTPUT_ROOT = Path("output")` — same tree as the cleaning notebook.

</details>


In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)

from pathlib import Path

from IPython.display import display

from heavy_machinery.config import load
from heavy_machinery.cleaning_phase.dataset_handoff import load_modelling_handoff
from heavy_machinery.cleaning_phase.missingness_resolution import load_modeling_frames
from heavy_machinery.cleaning_phase.validation import validate_unimputed_handoff, validate_imputed_frames
from heavy_machinery.modelling_phase.eda import screen_associations
from heavy_machinery.modelling_phase.diagnostic_accuracy import screen_diagnostic_accuracy
from heavy_machinery.modelling_phase.inferential import run_inferential_stage
from heavy_machinery.modelling_phase.marker_panel import run_marker_panel

OUTPUT_ROOT = Path("output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)  # do not wipe — reads cleaning outputs

#🟧🟧🟧 None = all years; e.g. [2025] for one cohort year

ANALYSIS_YEARS: list[int] | None = None

## 01. Load handoff

Requires `output/datasets/` parquets and `output/schema/schema_summary.csv` from cleaning §16.

`load_modelling_handoff` loads the **unimputed** cohort (`df`) for EDA, plus the ColSpec schema and which imputation method was run.

Next cell loads `output/cleaning/schema_validation.json` and validates the unimputed handoff only. Imputed draws are validated again in §04, immediately before multivariable modelling.


In [2]:
df, schema, IMPUTATION_METHOD = load_modelling_handoff(OUTPUT_ROOT)
#df.head(3)

Loaded unimputed cohort: 352 rows, 52 schema columns
Imputation method: MICE — inferential stage (§04) pools multiple imputed draws.


In [3]:
schema_validation = validate_unimputed_handoff(df, OUTPUT_ROOT)

✅ Pandera validated unimputed handoff (EDA cohort)


In [4]:
# 🟧🟧🟧 Copy-pasteable column names from the loaded cohort
#load("analysis").print_copy_pasteable_columns(df)

### 🎯 EDA


In [5]:
EDA_TARGETS = [
    'high_grade',
    #'progesterone_pos', 'brain_invasion', 'ki67_group', 'hist_necrosis'
    ]
# Binary targets only: {column: value coded as positive (1)}.
# Omit → auto-detect: True, else 1, else last sorted level (string binaries).
EDA_POSITIVE_CLASS = {
    #'progesterone_pos': True,
    #'brain_invasion': True,
    #'hist_necrosis': True,
}
EDA_PREDICTORS = [
    #🟧🟧🟧 Demographics
    'entry_year',
    'age',
    'sex',
    'tumor_episode',
    'tumor_location',
    'side',

    'male_sex',
    'irregular_margin',
    'skull_base_location',
    'midline',
    
    #🟧🟧🟧 Histological features
    #'who_grade',
    #'progesterone_pos',
    #'brain_invasion',
    #'hist_necrosis',
    
    #🟧🟧🟧 Imaging features
    'tumor_margin',
    'dural_tail',
    'mass_effect',
    'calcification',
    'cystic_component',
    'mri_necrosis',
    'hemorrhage',
    'hyperostosis',
    'cortical_destruction',
    
    'capsular_enhancement',
    'heterogeneous_enhancement',
    'dwi_hyperintensity',
    't2_hyperintensity',
    't1_hypointensity',
    'sinus_invasion',
    'transfalcine_extension',
    
    #🟧🟧🟧 Measurement features
    'meningioma_count',
    'perifocal_edema',
    'edema_volume_cm3',
    'max_diameter_cm',
    'tumor_volume',
    'adc_value',
    
    #🟧🟧🟧 Derived features
    #'high_grade',
    'multiple_meningiomas',
    #'ki67_mid',
    #'ki67_group',
    'edema_index',
    'venous_sinus_invasion',
    #'edema_index_ge1', 'edema_volume_ge3.64',  'tumor_volume_ge13.95', 'max_diameter_cm_gt6', 'max_diameter_cm_gt3'
    'edema_index_ge0.0617', 'edema_volume_ge4.76', 'tumor_volume_ge15.1', 'max_diameter_cm_ge3.81','adc_value_le0.72'
    ]

# Redundant variants of predictors already in the sweep: Youden-derived
# dichotomisations of continuous variables, and binary recodes of nominal
# parents. They render as exploratory (uncorrected) and stay out of the
# BH multiplicity family (spec 5.2).
EDA_REDUNDANT_VARIANTS = [
    'tumor_volume_ge15.1', 'adc_value_le0.72', 'max_diameter_cm_ge3.81',
    'edema_volume_ge4.76', 'edema_index_ge0.0617',
    'male_sex', 'irregular_margin', 'skull_base_location',
    'venous_sinus_invasion', 'midline',
]
EDA_FDR_FAMILY = [c for c in EDA_PREDICTORS if c not in EDA_REDUNDANT_VARIANTS]

### 📚 Literature-based multivariable models

In [6]:
# 📚 Literature-based multivariable models — published predictor sets.
# Each variant gets its own EPV bar, forest plot, VIF table, and interpretation.
# Format: (id, title, link, target, [predictors])
LITERATURE_MODEL_VARIANTS = [
    # Research work: Predicting the grade of meningiomas by clinical–radiological features: A comparison of precontrast and postcontrast MRI
    # Authors: Yuan Yao, Yifan Xu, Shihe Liu, Feng Xue, Bao Wang, Shanshan Qin, Xiubin Sun, Jingzhen He
    # Link: https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full
    (
        "yao_et_al_2022",
        "Yao et al. 2022 | precontrast / semantic MRI model",
        "https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full",
        "high_grade",
        [
            "male_sex",
            "irregular_margin",
            "cystic_component",
            "perifocal_edema",
            "dural_tail",
        ],
    ),

    # Research work: Preoperative Prediction of Intracranial Meningioma Grade Using Conventional CT and MRI
    # Authors: T. Amano et al.
    # Link: https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri
    (
        "amano_et_al_2021_expanded_proxy",
        "Amano et al. 2021 expanded proxy | conventional CT/MRI + tumor burden",
        "https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri",
        "high_grade",
        [
            "skull_base_location",
            "irregular_margin",
            "heterogeneous_enhancement",
            "perifocal_edema",
            "tumor_volume",
            "cortical_destruction",
        ],
    ),

    # Research work: The Role of Pre-Operative MRI for Prediction of High-Grade Intracranial Meningioma: A Retrospective Study
    # Authors: Kan Radeesri, Vitit Lekhavat
    # Link: https://journal.waocp.org/article_90552.html
    (
        "radeesri_lekhavat_2020",
        "Radeesri & Lekhavat 2020 | edema / necrosis MRI model",
        "https://journal.waocp.org/article_90552.html",
        "high_grade",
        [
            "perifocal_edema",
            "edema_volume_cm3",
            "mri_necrosis",
            "hemorrhage",
            "hyperostosis",
            "mass_effect",
        ],
    ),

    # Research work: Role of ADC values and ratios of MRI scan in differentiating typical, atypical and anaplastic meningiomas
    # Authors: M. Azeemuddin et al.
    # Link: https://pubmed.ncbi.nlm.nih.gov/30317276/
    (
        "azeemuddin_et_al_2018",
        "Azeemuddin et al. 2018 | diffusion-augmented MRI model",
        "https://pubmed.ncbi.nlm.nih.gov/30317276/",
        "high_grade",
        [
            "adc_value",
            "dwi_hyperintensity",
            "skull_base_location",
            "irregular_margin",
            "perifocal_edema",
            "heterogeneous_enhancement",
            "sex",
        ],
    ),

    # Research work: Diagnostic nomogram model for predicting preoperative pathological grade of meningioma
    # Authors: Shijun Peng, Zhihua Cheng, Zhilin Guo
    # Link: https://tcr.amegroups.org/article/view/55552/html
    (
        "peng_cheng_guo_2021",
        "Peng, Cheng & Guo 2021 | interface / invasion model",
        "https://tcr.amegroups.org/article/view/55552/html",
        "high_grade",
        [
            "skull_base_location",
            "irregular_margin",
            "venous_sinus_invasion",
            "cortical_destruction",
            "mass_effect",
            "edema_volume_cm3",
            "hyperostosis",
        ],
    ),
]

### 🧪 Experimental multivariable models


In [7]:
# 🧪 Experimental multivariable models — your own predictor sets (independent of EDA_PREDICTORS).
# Add as many as you need. Each row is one model: (id, title, link, target, [predictors]).
# Grouping in the report follows this list, not the model id string.

EXPERIMENTAL_MODEL_VARIANTS = [
    (
        "experimental_model_1",
        "model 1 | high grade",
        "",
        "high_grade",
        [
            'cystic_component',
            'cortical_destruction',
            'dural_tail',
            'tumor_volume_ge15.1',
            'edema_volume_cm3_ge4.76',
            'hyperostosis',
            'mass_effect',
            'adc_value_le0.72',
            'irregular_margin',
        ],
    ),
    (
        "experimental_model_2",
        "model 2 | high grade",
        "",
        "high_grade",
        [
            'dwi_hyperintensity',
            'sex',
            'heterogeneous_enhancement',
            'hemorrhage',
            'venous_sinus_invasion',
            'age',
            'calcification',
            't2_hyperintensity',
            't1_hypointensity',
            'transfalcine_extension',

        ],
    ),
]

In [8]:
_analysis = load("analysis")

EDA_TARGETS, EDA_PREDICTORS = _analysis.resolve_eda(df, EDA_TARGETS, EDA_PREDICTORS)

INFERENTIAL_MODEL_VARIANTS = _analysis.resolve_inferential_variants(
    df,
    LITERATURE_MODEL_VARIANTS,
    EXPERIMENTAL_MODEL_VARIANTS,
)
INFERENTIAL_TARGETS = _analysis.resolve_inferential_targets(df, INFERENTIAL_MODEL_VARIANTS)
# Binary inferential targets only: {column: value coded as positive (1)}. Omit → auto-detect.
INFERENTIAL_POSITIVE_CLASS = {}

## 03. EDA on unimputed data

`eda.screen_associations` and `diagnostic_accuracy.screen_diagnostic_accuracy`.



Targets can be **binary**, **continuous**, **ordinal**, or **nominal** (from schema). The test depends on both outcome and predictor types — e.g. ordinal outcome × nominal predictor → χ²; continuous outcome × nominal predictor → Kruskal–Wallis.

| target kind   | continuous / count predictor | ordinal predictor | nominal / binary predictor |
|---------------|------------------------------|-------------------|----------------------------|
| binary        | Mann–Whitney U               | Spearman ρ        | χ² / Fisher                |
| continuous    | Spearman ρ                   | Spearman ρ        | Kruskal–Wallis             |
| ordinal       | Spearman ρ                   | Spearman ρ        | χ²                         |
| nominal       | Kruskal–Wallis               | χ²                | χ²                         |

`POSITIVE_CLASS` applies only to **binary** targets. Multivariable logistic (§16) remains **binary outcomes only**.

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [9]:
assoc = screen_associations(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    fdr_family=EDA_FDR_FAMILY,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
    )

diag_acc = screen_diagnostic_accuracy(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

#assoc[assoc['fdr_significant']]

In [10]:
#🟧🟧🟧 Full table
#assoc

## 04. Multivariable modelling on imputed data

Load imputed dataset(s), Pandera-validate immediately before fitting, then `inferential.run_inferential_stage` (Rubin-pooled across MICE draws).

Writes per-variant tables and forest plots under `output/inferential/`, plus Streamlit JSON under `output/inferential/model_artifacts/`. Re-running clears stale variant files first.


In [11]:
imputed_frames = load_modeling_frames(OUTPUT_ROOT)
validate_imputed_frames(schema_validation, imputed_frames)

✅ Pandera validated 20 MICE imputed draws


In [12]:
full_inferential_table = run_inferential_stage(
    schema,
    imputed_frames=imputed_frames,
    targets=INFERENTIAL_TARGETS,
    variants=INFERENTIAL_MODEL_VARIANTS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    output_root=OUTPUT_ROOT,
    )
#full_inferential_table

## 04.5 · Marker panel — the two study aims

Takes the binary columns already in `df` and answers two questions on one cohort, writing everything to `output/panel/`:

1. **Which single sign argues hardest for the outcome?** Ranked by positive likelihood ratio.
2. **Does a combination beat any single sign?** Every pair scored and bootstrap-corrected for winner's curse, plus the count score, the fitted models, and MICE stability.

It creates no columns and searches for no cut-points — the threshold flags already exist by the time it runs.

| To change | Edit |
|---|---|
| which measurements get a cut-point searched | `meningioma-thresholder.ipynb` §02, `METRICS` |
| which cut-point is baked into a column | `meningioma-cleaning.ipynb`, `DERIVATIONS` |
| which binary columns this section uses | `MARKERS_TO_EXCLUDE` below |

Accuracy is computed on **observed** data, not the MICE draws: imputing a sign would report the accuracy of a finding nobody saw. The draws are used as a stability check instead.

In [13]:
# 🟧 FILL IN — predictors to keep out of the panel.
#    There is no pick-list: the panel takes every binary predictor the EDA
#    accuracy table carries for this target, and this set is the only lever.
#    Empty set() on purpose: bare {} is an empty dict, not an empty set.
MARKERS_TO_EXCLUDE: set[str] = set()
#    candidates: "sex_male", "hist_necrosis", "progesterone_pos"

panel_tables = run_marker_panel(
    df,
    target=EDA_TARGETS[0],
    accuracy_table=diag_acc,
    output_root=OUTPUT_ROOT,
    exclude=MARKERS_TO_EXCLUDE,
    variants=INFERENTIAL_MODEL_VARIANTS,
)

#display(panel_tables["02_marker_panel_reading_view"])
#display(panel_tables["09_selection_correction"])
#display(panel_tables["13_model_reading_view"])

## 05. Build report.html

`config/report_settings.py` + `report`. Assembles a self-contained HTML report from artifacts already in `output/` (DDA from cleaning, EDA, inferential). Launch the calculator separately: `streamlit run app.py` (reads `output/inferential/model_artifacts/`).



Builds `report.html` from artifacts already in `output/` (DDA from cleaning, EDA, inferential).
Edit the settings cell, then run both cells.

- **`REPORT_TITLE` / `REPORT_AUTHOR`** — shown on the cover.
- **`REPORT_PATH`** — where to write the HTML file.
- **`analysis_years`** — optional cohort label suffix on the title (from §03).
- **Module** — `load("report_settings")` → `config/report_settings.py` → `report`.


In [14]:
REPORT_TITLE = "Non-invasive radiological biomarkers of meningiomas as a prognostic tool for predicting tumor histological grade"
REPORT_AUTHOR = "Doc Arturs Balodis, Sigita Zālīte, Roberts Tumeļkāns, Valērija Aksjonova, Elizabete Stankeviča, Andris Zaguzovs"
REPORT_PATH = OUTPUT_ROOT / "report" / "report.html"

In [15]:
_report = load("report_settings")
_report.run_report(
    output_root=OUTPUT_ROOT,
    report_title=REPORT_TITLE,
    report_author=REPORT_AUTHOR,
    report_path=REPORT_PATH,
    analysis_years=ANALYSIS_YEARS,
    eda_targets=EDA_TARGETS,
)
_report.print_output_summary(OUTPUT_ROOT)

Report written: /Users/andriszaguzovs/TheLibraryOfCode/meningioma-atypier/output/report/report.html

📦 Pipeline outputs — /Users/andriszaguzovs/TheLibraryOfCode/meningioma-atypier/output
════════════════════════════════════════════════════════════════════════
📁 323 files · 39.8 MB total

🧹 Cleaning                  6 files · 113.2 KB  (5 csv, 1 json)
📋 Schema                    1 files ·   2.3 KB  (1 csv)
💾 Model datasets            3 files ·  90.0 KB  (1 json, 2 parquet)
📊 DDA                     136 files ·   9.7 MB  (6 csv, 130 svg) — figures: 43, figures_bivariate: 87, tables: 6
🕳️ Missingness              58 files ·   3.5 MB  (30 csv, 4 json, 20 parquet, 2 png, 2 svg) — figures: 2, mice: 54, tables: 2
🔬 EDA                      43 files ·   2.8 MB  (2 csv, 41 svg) — figures: 41, tables: 2
🧮 Multivariable            59 files ·   2.3 MB  (16 csv, 14 json, 29 svg) — figures: 29, model_artifacts: 7, tables: 23
🧾 Report                    1 files ·  20.5 MB  (1 html)
📁 Panel           